# Bot TDI — Finanzas, política y cultura general de Argentina

## Todo el proyecto corre desde este único notebook

Bot de Telegram que consulta datos públicos de Argentina (dólar, riesgo país, inflación, presidentes históricos) usando la API de [ArgentinaDatos](https://argentinadatos.com/docs/).

**Estado (parte 2 de 3):** configuración, historial en CSV, `/start` con aviso al admin, `/dolar`, `/riesgo`, `/historial` y `/ayuda` con botones inline.
Falta la parte 3: presidentes, inflación con gráfico y respuestas a audio, imagen, video, documento y texto libre.

## Cómo funciona el flujo

```text
Persona → Telegram → API de Telegram → polling del notebook → handlers → respuesta
```

- El **kernel de Jupyter es el proceso del bot**: mientras la última celda esté ejecutándose, el bot responde. Si se detiene el kernel, el bot deja de contestar.
- Usamos **polling**: el programa le pregunta a Telegram cada pocos segundos si hay mensajes nuevos.
- Un token solo admite **un proceso haciendo polling**. Si hay dos a la vez (por ejemplo `main.py` y este notebook, o dos compañeros con el mismo token), Telegram devuelve el error 409 y los mensajes se reparten o se pierden.

## Cómo trabajar con este notebook

1. Ejecutá las celdas **en orden, de arriba hacia abajo** (`Shift+Enter`).
2. La **última celda arranca el bot y queda ocupada**; no termina hasta que la interrumpas.
3. Para agregar un handler nuevo: **Interrupt** en la celda de arranque → ejecutá la celda nueva → volvé a ejecutar la celda de arranque.
4. Si cambiás algo de la configuración o del `.env`, reiniciá el kernel (**Restart**) y ejecutá todo de nuevo.

## ⚠️ Cuidado al modificar un handler ya ejecutado

`telebot` guarda los handlers en una lista y, para cada mensaje, ejecuta **el primero que coincide**. Si cambiás el código de un handler y volvés a ejecutar su celda, el handler **viejo sigue registrado primero** y es el que responde. Después de modificar un handler: **Restart** del kernel y ejecutar todo de nuevo.

## Dependencias

Se instalan una sola vez en el entorno virtual. La celda queda comentada para que no se reinstale en cada ejecución; si hace falta, sacale el `#` y ejecutala.

In [ ]:
# %pip install pyTelegramBotAPI python-dotenv requests matplotlib

## 1. Configuración y creación del bot

Qué hace esta celda:

- Importa las librerías que usa el notebook.
- Lee `BOT_TOKEN` y `CHAT_ID_ADMIN` desde el archivo `.env` (junto a este notebook). **El token nunca se escribe en el código**, para que no termine en GitHub.
- Si falta el token, corta con un mensaje claro en vez de fallar más adelante.
- Crea el objeto `bot` con `parse_mode="Markdown"`: así `*negrita*` y `_cursiva_` funcionan en todos los mensajes sin repetirlo.
- Define `escapar_md()`, un ayudante para texto que viene de afuera (nombres de usuario, por ejemplo). Si un nombre trae un `_` o un `*`, sin escapar rompería el formato del mensaje.

In [ ]:
import csv
import os
import threading
import time
from datetime import date, datetime, timedelta
from pathlib import Path

import requests
import telebot
from dotenv import load_dotenv
from telebot import types
from telebot.apihelper import ApiTelegramException

load_dotenv(override=False)
BOT_TOKEN = os.getenv("BOT_TOKEN")
CHAT_ID_ADMIN = os.getenv("CHAT_ID_ADMIN")

if not BOT_TOKEN:
    raise RuntimeError("Falta BOT_TOKEN en el archivo .env (junto a este notebook).")

bot = telebot.TeleBot(BOT_TOKEN, parse_mode="Markdown")


def escapar_md(texto: str) -> str:
    """Escapa los caracteres que rompen el Markdown clásico de Telegram."""
    for c in ("_", "*", "`", "["):
        texto = texto.replace(c, "\\" + c)
    return texto


print("Bot creado. Todavía no está escuchando mensajes.")

## 2. Historial de interacciones en CSV

La consigna pide registrar cada interacción en un `.csv` con estas columnas: **timestamp**, **chat_id**, **nombre** y **mensaje**. Ese archivo después se usa para el comando `/historial` y para el notebook de análisis.

Cómo está armado:

- `registrar_interaccion()` agrega una fila al CSV. Si el archivo no existe, lo crea con el encabezado.
- `descripcion_mensaje()` decide qué texto guardar: el texto del mensaje si lo hay, o una etiqueta (`[audio]`, `[imagen]`, `[video]`, `[documento]`) cuando no es texto.
- `listener_historial()` es un **update listener**: `telebot` lo ejecuta con cada tanda de mensajes nuevos, **antes** de los handlers. Así se registra todo sin repetir código en cada comando.
- El `Lock` evita que dos mensajes simultáneos escriban el archivo al mismo tiempo, porque `telebot` atiende cada mensaje en un hilo distinto.

**Limitación:** el listener recibe solo *mensajes*, no los toques de botones inline. Cuando armemos los botones, los registramos desde su propio handler.

El archivo se guarda en `historial/historial.csv` (relativo a la carpeta del notebook) y está en el `.gitignore`, porque tiene nombres e IDs de usuarios.

In [ ]:
RUTA_CSV = Path("historial") / "historial.csv"
CAMPOS = ["timestamp", "chat_id", "nombre", "mensaje"]
IGNORAR_EN_HISTORIAL = {"/historial", "[botón] historial"}  # no se muestran dentro del propio historial
_lock = threading.Lock()


def registrar_interaccion(chat_id, nombre, mensaje):
    RUTA_CSV.parent.mkdir(exist_ok=True)
    with _lock:
        existe = RUTA_CSV.exists()
        with open(RUTA_CSV, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            if not existe:
                writer.writerow(CAMPOS)
            writer.writerow(
                [datetime.now().isoformat(timespec="seconds"), chat_id, nombre, mensaje]
            )


def descripcion_mensaje(m):
    if m.text:
        return m.text
    if m.voice or m.audio:
        return "[audio]"
    if m.photo:
        return "[imagen]"
    if m.video or m.video_note:
        return "[video]"
    if m.document:
        return "[documento]"
    return "[otro tipo de mensaje]"


def listener_historial(mensajes):
    for m in mensajes:
        if m.from_user is None:
            continue
        registrar_interaccion(m.chat.id, m.from_user.first_name, descripcion_mensaje(m))


bot.set_update_listener(listener_historial)
print("Registro de historial activado.")

## 3. Comando `/start`

`/start` cumple dos requisitos de la consigna:

- **Saluda al usuario por su nombre** (`message.from_user.first_name`) y lo guía hacia `/ayuda`. El texto usa *negrita* y un emoji.
- **Responde en más de un chat**: además de contestarle a quien escribió, `avisar_al_admin()` le manda un mensaje al chat del administrador (`CHAT_ID_ADMIN` del `.env`) con quién inició el bot.

Detalles:

- El aviso al admin va **después** del saludo y dentro de un `try/except`: si el chat del admin no existe, el ID está mal o el admin nunca habló con el bot, el usuario igual recibe su respuesta.
- Un bot no puede escribirle a alguien que nunca inició una conversación con él, así que el admin tiene que haber hecho `/start` al menos una vez.
- Para ver el aviso en la demo, el `/start` tiene que venir de **otra cuenta**; si lo hacés con la misma cuenta del admin, ves los dos mensajes en el mismo chat.

In [ ]:
def avisar_al_admin(message):
    """Segundo chat de la respuesta: avisa al admin que alguien inició el bot."""
    if not CHAT_ID_ADMIN:
        return
    u = message.from_user
    username = f"@{u.username}" if u.username else "sin username"
    texto = (
        "🔔 *Nuevo /start*\n\n"
        f"Nombre: {escapar_md(u.full_name)}\n"
        f"Usuario: {escapar_md(username)}\n"
        f"Chat ID: `{message.chat.id}`"
    )
    try:
        bot.send_message(int(CHAT_ID_ADMIN), texto)
    except (ApiTelegramException, ValueError):
        pass  # si falla el aviso, que no afecte la respuesta al usuario


@bot.message_handler(commands=["start"])
def comando_start(message):
    nombre = message.from_user.first_name or "che"
    texto = (
        f"¡Hola {nombre}! 👋\n\n"
        "Soy tu bot de *finanzas, política y cultura general* de Argentina.\n"
        "Puedo contarte sobre la cotización del dólar, presidentes históricos, "
        "riesgo país, inflación y más.\n\n"
        "Escribí /ayuda para ver todo lo que puedo hacer."
    )
    bot.reply_to(message, texto)
    avisar_al_admin(message)

## 4. Cotización del dólar (`/dolar`)

Primera consulta a una **API externa**: [ArgentinaDatos](https://argentinadatos.com/docs/operations/get-cotizaciones-dolares.html).

Cómo funciona:

- `obtener_cotizaciones()` hace un `GET` y devuelve la lista completa, o `None` si falla (sin internet, timeout, error del servidor). El `timeout=10` evita que el bot se cuelgue esperando para siempre.
- **Ojo con la forma de los datos:** el endpoint devuelve el **historial completo de todas las casas mezclado**, no solo la cotización de hoy. Por eso `obtener_ultima_cotizacion()` filtra una casa (oficial, blue) y se queda con el registro de **fecha más reciente** (`max(..., key=fecha)`). Si tomáramos el primero, saldría un dato de 2011.
- `texto_cotizacion_dolar()` arma el mensaje con *negrita*, cursiva y emoji, y avisa si la API no responde.
- El `send_chat_action("typing")` muestra "escribiendo..." mientras se consulta la API (requisito de la consigna).
- Es el mismo texto que van a usar el comando `/dolar` y el botón del `/ayuda`.

In [ ]:
NOMBRES_CASAS = {
    "oficial": "Dólar Oficial",
    "blue": "Dólar Blue",
}
URL_DOLARES = "https://api.argentinadatos.com/v1/cotizaciones/dolares"


def obtener_cotizaciones():
    """Lista completa de cotizaciones (todas las casas y fechas), o None si falla."""
    try:
        response = requests.get(URL_DOLARES, timeout=10)
        response.raise_for_status()
        return response.json()
    except requests.RequestException:
        return None


def obtener_ultima_cotizacion(cotizaciones, casa):
    """De toda la lista, el registro más reciente de una casa puntual."""
    de_esa_casa = [c for c in cotizaciones if c["casa"] == casa]
    if not de_esa_casa:
        return None
    return max(de_esa_casa, key=lambda c: c["fecha"])


def texto_cotizacion_dolar() -> str:
    cotizaciones = obtener_cotizaciones()
    if cotizaciones is None:
        return "😕 No pude consultar la cotización ahora. Probá de nuevo en un rato."

    lineas = ["💵 *Cotización del dólar*\n"]
    for casa, nombre in NOMBRES_CASAS.items():
        item = obtener_ultima_cotizacion(cotizaciones, casa)
        if item is None:
            lineas.append(f"*{nombre}*: no disponible")
            continue
        lineas.append(
            f"*{nombre}*: compra ${item['compra']} / venta ${item['venta']} "
            f"_(al {item['fecha']})_"
        )
    return "\n".join(lineas)


@bot.message_handler(commands=["dolar"])
def comando_dolar(message):
    bot.send_chat_action(message.chat.id, "typing")
    bot.reply_to(message, texto_cotizacion_dolar())

## 5. Riesgo país (`/riesgo`)

Dos consultas a la misma API:

- `/riesgo-pais/ultimo` → el valor más reciente.
- `/riesgo-pais` → la serie histórica, que se usa **solo para comparar** con hace 30 días.

Para que el dato se entienda, el mensaje incluye:

- **Qué significa:** el extra de interés que piden los inversores para prestarle a Argentina, comparado con EE.UU.
- **El número traducido:** 100 puntos básicos = 1 punto porcentual, así que `607` equivale a 6,07 puntos porcentuales.
- **La tendencia a 30 días**, calculada con la serie histórica.

La comparación es un **extra**: `_dato_hace_30_dias()` devuelve `None` ante cualquier problema, y en ese caso el mensaje sale igual, sin esa última línea.

In [ ]:
URL_RIESGO_ULTIMO = "https://api.argentinadatos.com/v1/finanzas/indices/riesgo-pais/ultimo"
URL_RIESGO_SERIE = "https://api.argentinadatos.com/v1/finanzas/indices/riesgo-pais"


def obtener_riesgo_pais():
    """Último valor publicado: {"valor": int, "fecha": "YYYY-MM-DD"}, o None si falla."""
    try:
        response = requests.get(URL_RIESGO_ULTIMO, timeout=10)
        response.raise_for_status()
        return response.json()
    except requests.RequestException:
        return None


def _dato_hace_30_dias(fecha_ultima: str):
    """Último valor publicado hasta 30 días antes de `fecha_ultima`, o None."""
    try:
        response = requests.get(URL_RIESGO_SERIE, timeout=10)
        response.raise_for_status()
        serie = response.json()
        limite = (date.fromisoformat(fecha_ultima) - timedelta(days=30)).isoformat()
        anteriores = [d for d in serie if d["fecha"] <= limite]
        if not anteriores:
            return None
        return max(anteriores, key=lambda d: d["fecha"])
    except (requests.RequestException, ValueError, KeyError, TypeError):
        return None


def texto_riesgo_pais() -> str:
    dato = obtener_riesgo_pais()
    if dato is None:
        return "😕 No pude consultar el riesgo país ahora. Probá de nuevo en un rato."

    valor = dato["valor"]
    puntos_porcentuales = f"{valor / 100:.2f}".replace(".", ",")

    lineas = [
        "📊 *Riesgo país*\n",
        f"*{valor}* puntos básicos _(al {dato['fecha']})_\n",
        "💡 *¿Qué significa?*",
        "Es el extra de interés que los inversores piden para prestarle plata a "
        "Argentina, comparado con lo que paga EE.UU. (considerado el deudor más seguro).",
        f"En este caso, {puntos_porcentuales} puntos porcentuales más "
        "(100 puntos básicos equivalen a 1 punto porcentual).",
        "Si baja, los inversores perciben menos riesgo; si sube, perciben más.",
    ]

    anterior = _dato_hace_30_dias(dato["fecha"])
    if anterior:
        diferencia = valor - anterior["valor"]
        if diferencia > 0:
            tendencia = f"🔺 Subió {diferencia} puntos"
        elif diferencia < 0:
            tendencia = f"🔻 Bajó {abs(diferencia)} puntos"
        else:
            tendencia = "➡️ Se mantuvo igual"
        lineas.append(
            f"\n{tendencia} en los últimos 30 días "
            f"(era {anterior['valor']} el {anterior['fecha']})."
        )
    return "\n".join(lineas)


@bot.message_handler(commands=["riesgo"])
def comando_riesgo(message):
    bot.send_chat_action(message.chat.id, "typing")
    bot.reply_to(message, texto_riesgo_pais())

## 6. Historial de consultas (`/historial`)

Lee el CSV de la sección 2 y le muestra a cada persona **sus últimas 5 consultas**.

- `ultimas_consultas()` filtra por `chat_id` (cada usuario ve solo lo suyo) y descarta los pedidos del propio historial (`IGNORAR_EN_HISTORIAL`), porque listar "/historial" dentro del historial no aporta nada.
- Se usa el mismo `Lock` que al escribir, para no leer el archivo mientras otro mensaje lo está modificando.
- `escapar_md()` evita que un mensaje con `_` o `*` rompa el formato.
- Cuando el usuario todavía no hizo ninguna consulta, responde con un mensaje amable en lugar de una lista vacía.

In [ ]:
def ultimas_consultas(chat_id, cantidad: int = 5) -> list:
    """Últimas `cantidad` consultas de ese chat (de la más vieja a la más nueva)."""
    if not RUTA_CSV.exists():
        return []
    with _lock:
        with open(RUTA_CSV, newline="", encoding="utf-8") as f:
            filas = [
                fila
                for fila in csv.DictReader(f)
                if fila["chat_id"] == str(chat_id)
                and fila["mensaje"] not in IGNORAR_EN_HISTORIAL
            ]
    return filas[-cantidad:]


def texto_historial(chat_id) -> str:
    filas = ultimas_consultas(chat_id)
    if not filas:
        return "📜 *Historial de consultas*\n\nTodavía no hiciste ninguna consulta."

    lineas = ["📜 *Tus últimas consultas*\n"]
    for fila in filas:
        lineas.append(f"• `{fila['timestamp']}` — {escapar_md(fila['mensaje'])}")
    return "\n".join(lineas)


@bot.message_handler(commands=["historial"])
def comando_historial(message):
    bot.reply_to(message, texto_historial(message.chat.id))

## 7. `/ayuda` y botones inline

Cumple dos requisitos de la consigna: el **listado de comandos principales** y **atajos con botones inline**.

Cómo funcionan los botones:

1. `teclado_ayuda()` arma un `InlineKeyboardMarkup` con un botón por función. Cada botón lleva un `callback_data`: una palabra clave (`"dolar"`, `"riesgo_pais"`...) que identifica cuál se tocó.
2. `/ayuda` envía el listado de comandos y adjunta ese teclado con `reply_markup`.
3. Cuando alguien toca un botón, Telegram **no manda un mensaje**: manda un *callback query*. Lo atiende `boton()`, que decide qué responder según el `callback_data`.
4. `bot.answer_callback_query()` le saca el "reloj de carga" al botón; si se omite, el botón parece colgado.

Dos detalles:

- Los toques de botón **no pasan por el listener** del historial (solo ve mensajes), por eso `boton()` los registra a mano como `[botón] dolar`, etc.
- 🕰️ **Presidentes** y 📈 **Inflación** ya tienen botón, pero responden "en construcción" hasta la parte 3. Para sumarlos habrá que agregarlos a `FUNCIONES_BOTON`.

In [ ]:
def teclado_ayuda():
    teclado = types.InlineKeyboardMarkup(row_width=1)
    teclado.add(
        types.InlineKeyboardButton("💵 Cotización del dólar", callback_data="dolar"),
        types.InlineKeyboardButton("📊 Riesgo país", callback_data="riesgo_pais"),
        types.InlineKeyboardButton("🕰️ Presidentes históricos", callback_data="presidentes"),
        types.InlineKeyboardButton("📈 Inflación (últimos 6 meses)", callback_data="inflacion"),
        types.InlineKeyboardButton("📜 Historial de consultas", callback_data="historial"),
    )
    return teclado


@bot.message_handler(commands=["ayuda"])
def comando_ayuda(message):
    texto = (
        "📋 *Comandos disponibles:*\n\n"
        "/start — iniciar la conversación\n"
        "/ayuda — ver esta ayuda\n"
        "/dolar — cotización del dólar oficial y blue\n"
        "/riesgo — riesgo país, con contexto\n"
        "/historial — tus últimas 5 consultas\n\n"
        "También podés tocar uno de estos atajos 👇"
    )
    bot.send_message(message.chat.id, texto, reply_markup=teclado_ayuda())


# callback_data -> función que devuelve el texto de la respuesta
FUNCIONES_BOTON = {
    "dolar": texto_cotizacion_dolar,
    "riesgo_pais": texto_riesgo_pais,
}


@bot.callback_query_handler(func=lambda call: True)
def boton(call):
    bot.answer_callback_query(call.id)  # saca el "reloj de carga" del botón
    chat_id = call.message.chat.id
    registrar_interaccion(chat_id, call.from_user.first_name, f"[botón] {call.data}")

    bot.send_chat_action(chat_id, "typing")
    if call.data == "historial":
        texto = texto_historial(chat_id)
    elif call.data in FUNCIONES_BOTON:
        texto = FUNCIONES_BOTON[call.data]()
    else:
        texto = "🚧 Esta función todavía está en construcción."
    bot.send_message(chat_id, texto)

> A partir de acá, para sumar un handler nuevo (parte 3), **va antes de la celda de arranque**. Y los handlers generales (texto libre, audio, imagen...) tienen que registrarse **después** de todos los comandos: `telebot` ejecuta el primero que coincide.

## 8. Arrancar el bot (siempre la última celda)

Esta celda **queda ejecutándose** mientras el bot esté activo. Para detenerlo: **Interrupt** (⏹) en la barra de la celda.

Qué hace cada línea:

- `remove_webhook()`: Telegram no permite recibir actualizaciones por webhook y por polling a la vez. Si el bot usó webhook antes, esto lo limpia.
- `skip_pending=True`: ignora los mensajes que se acumularon mientras el bot estaba apagado, para que no responda de golpe una tanda vieja.
- `timeout` y `long_polling_timeout`: cuánto espera cada consulta a Telegram antes de reintentar.
- `allowed_updates=["message", "callback_query"]`: qué tipos de novedades recibe. **`callback_query` es lo que hace funcionar los botones inline**; si se omite, los botones no hacen nada. (El notebook de ejemplo del profe usa solo `message` porque no tenía botones.)

Antes de volver a ejecutar esta celda, interrumpí la anterior o reiniciá el kernel.

In [ ]:
bot.remove_webhook()
time.sleep(1)

print("Bot escuchando con polling. Para detenerlo: Interrupt Kernel.")
bot.infinity_polling(
    skip_pending=True,
    timeout=20,
    long_polling_timeout=20,
    allowed_updates=["message", "callback_query"],
)

## Cómo probar esta parte

1. Ejecutá todas las celdas en orden; la de arranque (sección 8) siempre la última.
2. `/start` → saludo por nombre (y aviso al admin si lo hace otra cuenta).
3. `/ayuda` → listado de comandos y 5 botones. Tocá 💵, 📊 y 📜.
4. `/dolar` y `/riesgo` → deben mostrar datos reales con su fecha.
5. `/historial` → tus últimas consultas, incluidos los toques de botón (`[botón] dolar`).
6. Abrí `historial/historial.csv` y confirmá que cada acción agregó una fila.
7. Apagá el wifi un instante y probá `/dolar`: tiene que responder con el mensaje de error, sin romperse.